# # 使用贝叶斯优化得到的最佳超参数进行模型训练
#
# 本notebook旨在利用之前贝叶斯优化得到的最佳超参数组合，重新进行一次完整的模型训练、验证和测试。
# 我们将尽可能复用项目中的现有代码模块。


In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR, ReduceLROnPlateau
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm # 使用notebook版本的tqdm
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix

# 导入项目中的模块
from config import load_config, save_config, CONFIG # 导入CONFIG以获取一些默认值或结构
from data import load_data
from models import get_model
from utils.visualization import visualize_training_curves, visualize_confusion_matrix, visualize_dataset_distribution


In [ ]:
# ## 2. 定义和加载配置 (重点：确保使用MAT加载格式)
#
# 首先，加载基础配置。然后，我们将**强制指定MAT文件路径和患者分割信息**，并用贝叶斯优化得到的最佳超参数覆盖它。

# %%
# 加载默认配置
cfg = load_config()

# **强制使用MAT数据加载格式的关键配置**
# !! 请确保以下路径和ID列表与您的设置一致 !!
cfg['mat_file_path'] = "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat" # <--- 修改这里，指向您的TRAIN38.mat文件
# 如果您的 .mat 文件名不同，请相应修改。
# 如果 cfg['mat_file_path'] 被正确设置并指向一个存在的 .mat 文件，
# config.py 中的 _auto_detect_data_format 会自动将 label_format 设置为 'mat'
# 并且 data/__init__.py 中的 load_data 会选择 mat_loader.py 中的加载逻辑。

# 从 config.py 获取默认的患者ID分割，或者在此处硬编码
# 这些ID用于 process_train38_data 函数中分割数据集
default_dataset_split = CONFIG.get('dataset_split', { # 从原始CONFIG中获取，避免被load_config可能加载的json覆盖（如果json中没有此项）
    'train_patients': [28, 5, 25, 30, 34, 32, 33, 11, 12, 20, 29, 17, 37, 7, 26, 1, 36, 14, 19, 3, 35, 31, 22, 8],
    'val_patients': [4, 24, 9, 15, 16, 18, 2],
    'test_patients': [38, 6, 21, 13, 10, 23, 27]
})
cfg['dataset_split'] = default_dataset_split

# 为了确保不意外使用旧的数据目录格式，可以将data_dirs设置为空或None
cfg['data_dirs'] = {
    'train_dir': None,
    'test_dir': None,
    'val_dir': None
}

print(f"确认配置 mat_file_path: {cfg.get('mat_file_path')}")
print(f"确认配置 dataset_split (部分): {list(cfg.get('dataset_split', {}).get('train_patients', [])[:3])}...") # 打印前几个训练患者ID以确认



In [ ]:
# ## 3. 定义最佳超参数
#
# 从您的日志中提取最佳试验的参数。
# **请将以下 `best_hyperparams_from_log` 字典替换为您日志中 `Best is trial ... with value: ...` 后面跟着的参数字典。**

# %%
best_hyperparams_from_log = {
    'learning_rate': 9.191261837889327e-05,
    'weight_decay': 0.0005175833650131985,
    'optimizer': 'adamw',
    'dropout_rate': 0.19317216698770392,
    'activation': 'gelu',
    'lr_scheduler_type': 'step', # Optuna中可能叫 'lr_scheduler'
    'model_type': 'base_mlp',
    'hidden_units': cfg.get('hidden_units', [4096, 4096, 4096, 4096]), # 使用cfg的默认值或您优化的值
    'lr_step_size': 4, # 对应日志中的 'step_size'
    'lr_gamma': 0.1590396939768399, # 对应日志中的 'step_gamma'
}

# 更新配置
for key, value in best_hyperparams_from_log.items():
    if key == 'learning_rate':
        cfg['lr'] = value
    elif key == 'lr_scheduler':
        cfg['lr_scheduler_type'] = value
    elif key == 'step_size':
        cfg['lr_step_size'] = value # 直接使用，假设 StepLR
    elif key == 'step_gamma':
         cfg['lr_gamma'] = value
    else:
        cfg[key] = value

cfg['epochs'] = cfg.get('epochs', 30)
cfg['batch_size'] = cfg.get('batch_size', 128)
cfg['val_epochs'] = cfg.get('val_epochs', 3)
cfg['use_lr_scheduler'] = True

print("更新后的训练相关配置:")
for key, value in cfg.items():
    if key in best_hyperparams_from_log or key in ['lr', 'lr_scheduler_type', 'lr_step_size', 'lr_gamma', 'epochs', 'batch_size', 'val_epochs', 'use_lr_scheduler', 'hidden_units']:
        print(f"  {key}: {value}")

In [ ]:
# ## 4. 设置设备 (CPU/GPU)

# %%
device = torch.device(f"cuda:{cfg['device']}" if cfg['device'] >= 0 and torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.manual_seed(cfg['random_seed'])
if device.type == 'cuda':
    torch.cuda.manual_seed_all(cfg['random_seed'])
np.random.seed(cfg['random_seed'])


In [ ]:
# ## 5. 加载数据 (使用MAT加载流程)
#
# 调用 `load_data`。由于 `cfg['mat_file_path']` 已设置，此函数将自动使用MAT文件加载逻辑。

# %%
# 运行 _auto_detect_data_format (load_config 内部会调用，这里是为了确保MAT路径正确后再次确认)
# 如果是从头开始运行此notebook，并且 mat_file_path 在上面已正确设置，
# load_config 时应该已经检测到了。为了更明确，可以手动调用内部函数（但不推荐直接调用私有函数）
# 或者依赖 load_data 打印的信息。
# cfg = load_config(cfg) # 重新加载配置以确保 _auto_detect_data_format 基于新的 mat_file_path 运行
#                        # 但这会重置其他参数，所以更好的方式是确保 mat_file_path 在 load_config 前设置
#                        # 或者像上面那样，先 load_config，再修改 mat_file_path，然后 load_data 会自行判断

print(f"准备使用配置中的 mat_file_path: {cfg['mat_file_path']} 加载数据...")
if not os.path.exists(cfg['mat_file_path']):
    raise FileNotFoundError(f"MAT文件未找到: {cfg['mat_file_path']}. 请在单元格2中设置正确的路径。")

dataset_dict, train_loader, val_loader, test_loader = load_data(cfg, mode='train')

# mat_loader.py 中的 load_and_process_data 会更新 config 中的 feature_dim 和 num_class
# 这里再次确认它们被正确传递或从 dataset_dict 获取
if 'feature_dim' in dataset_dict:
    cfg['feature_dim'] = dataset_dict['feature_dim']
    print(f"特征维度 (feature_dim) 从 dataset_dict 更新: {cfg['feature_dim']}")
# num_classes 在 process_train38_data 中计算并放入 dataset_dict
if 'num_classes' in dataset_dict:
    cfg['num_class'] = dataset_dict['num_classes']
    print(f"类别数量 (num_class) 从 dataset_dict 更新: {cfg['num_class']}")
else: # Fallback if not present, though mat_loader should provide it
    all_labels_in_sets = np.unique(np.concatenate([
        dataset_dict['train_labels'].flatten(), # .flatten() for one-hot encoded labels if argmax wasn't applied yet
        dataset_dict['test_labels'].flatten(),
        dataset_dict['val_labels'].flatten()
    ]))
    # BrainVoxelMatDataset maps labels 1-102 to 0-101
    # So num_class is the count of unique labels after mapping (e.g., 102 if all are present)
    # process_train38_data filters out background (label 0) before creating datasets
    # The labels in y_train, y_test, y_val from process_train38_data are 1-102 (if not one-hot).
    # BrainVoxelMatDataset then maps these to 0-101.
    # So, number of classes should be the number of unique original labels (e.g. 102).
    raw_unique_labels = np.unique(np.concatenate([
        np.argmax(dataset_dict['train_labels'], axis=1) if len(dataset_dict['train_labels'].shape) > 1 and dataset_dict['train_labels'].shape[1] > 1 else dataset_dict['train_labels'],
        np.argmax(dataset_dict['test_labels'], axis=1) if len(dataset_dict['test_labels'].shape) > 1 and dataset_dict['test_labels'].shape[1] > 1 else dataset_dict['test_labels'],
        np.argmax(dataset_dict['val_labels'], axis=1) if len(dataset_dict['val_labels'].shape) > 1 and dataset_dict['val_labels'].shape[1] > 1 else dataset_dict['val_labels']
    ]))
    cfg['num_class'] = len(raw_unique_labels) # This should be 102 if all classes (1-102) are present.
    print(f"类别数量 (num_class) 从数据标签推断: {cfg['num_class']}")


# 可选：可视化数据集样本数量（类别分布对于大型MAT文件可能不方便直接可视化，除非修改 visualization.py）
# 因为 visualize_dataset_distribution 设计上是基于 dataset_dict['train_labels'] 等直接是标签列表
# 而 MAT 加载后，dataset_dict 中的标签可能是 one-hot 编码或未经转换的。
# process_train38_data 内部有打印样本数量。
print(f"训练集样本数: {len(dataset_dict['train_samples'])}")
print(f"验证集样本数: {len(dataset_dict['val_samples'])}")
print(f"测试集样本数: {len(dataset_dict['test_samples'])}")


In [ ]:
# ## 6. 初始化模型
# (这部分与之前相同，依赖于cfg中的feature_dim, num_class等)

# %%
model_params = {
    'input_dim': cfg['feature_dim'],
    'hidden_dims': cfg['hidden_units'],
    'num_classes': cfg['num_class'],
    'dropout_rate': cfg['dropout_rate'],
    'activation': cfg['activation']
}

if cfg['model_type'] == 'deep_mlp':
    model_params['use_skip_connections'] = cfg.get('use_skip_connections', False)
elif cfg['model_type'] == 'residual_mlp':
    model_params['use_bottleneck'] = cfg.get('use_bottleneck', False)
    model_params['bottleneck_factor'] = cfg.get('bottleneck_factor', 0.5)

model = get_model(cfg['model_type'], **model_params)
model.to(device)

print(f"模型类型: {cfg['model_type']}")
print(f"模型输入维度: {cfg['feature_dim']}, 输出类别数: {cfg['num_class']}")
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"总参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")


In [ ]:
# ## 7. 定义优化器、学习率调度器和损失函数
# (这部分与之前相同)

# %%
# Optimizer
if cfg['optimizer'].lower() == 'adam':
    optimizer = optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
elif cfg['optimizer'].lower() == 'adamw':
    optimizer = optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
else:
    raise ValueError(f"不支持的优化器: {cfg['optimizer']}")

# Learning Rate Scheduler
scheduler = None
if cfg['use_lr_scheduler']:
    if cfg['lr_scheduler_type'].lower() == 'multistep': # config.py 中有多步调度的里程碑
        scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=cfg['lr_milestones'], gamma=cfg['lr_gamma'])
    elif cfg['lr_scheduler_type'].lower() == 'step': # 用于从日志中直接获取的 step_size 和 step_gamma
        scheduler = StepLR(optimizer, step_size=cfg.get('lr_step_size', 5), gamma=cfg.get('lr_gamma',0.1))
    elif cfg['lr_scheduler_type'].lower() == 'cosine':
        scheduler = CosineAnnealingLR(optimizer, T_max=cfg['epochs'], eta_min=cfg.get('lr_min', 1e-7)) # lr_min 对应 eta_min
    elif cfg['lr_scheduler_type'].lower() == 'plateau':
        scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=cfg['lr_gamma'], patience=cfg.get('lr_patience', 5), verbose=True) # lr_patience for plateau
    else:
        print(f"未知的学习率调度器类型: {cfg['lr_scheduler_type']}. 不使用调度器。")


# Loss Function
# BrainVoxelMatDataset 将标签从 1-102 映射到 0-101，背景（原始标签0）在 process_train38_data 中已被过滤
# 因此，不需要 ignore_index。
criterion = nn.CrossEntropyLoss()

print(f"优化器: {cfg['optimizer']}")
if scheduler:
    print(f"学习率调度器: {cfg['lr_scheduler_type']}")
print(f"损失函数: CrossEntropyLoss")


In [ ]:
# ## 8. 训练与验证循环
# (这部分与之前基本相同)

# %%
training_results = {
    'loss_list': [],
    'acc_list': [],
    'val_epoch_list': [],
    'val_acc_list': [],
    'val_f1_macro_list': [],
    'val_kappa_list': []
}

best_val_acc = 0.0
best_model_filename = f"{cfg.get('model_name', 'model')}_{cfg.get('dataset_name','dataset')}_best_mat_format.pth"
best_model_path = os.path.join(cfg.get('save_dir', './results'), best_model_filename)
os.makedirs(cfg.get('save_dir', './results'), exist_ok=True)


print(f"开始训练模型: {cfg['model_name']}")
print(f"总轮数 (Epochs): {cfg['epochs']}")
print(f"验证频率 (Val Epochs): {cfg['val_epochs']}")

for epoch in range(1, cfg['epochs'] + 1):
    model.train()
    epoch_loss = 0
    train_preds_epoch = []
    train_targets_epoch = []

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{cfg['epochs']} [Train]", leave=False)
    for batch_idx, (data, target) in enumerate(progress_bar):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        preds = torch.argmax(output, dim=1)
        train_preds_epoch.extend(preds.cpu().numpy())
        train_targets_epoch.extend(target.cpu().numpy())
        
        if batch_idx % 100 == 0:
             progress_bar.set_postfix(loss=loss.item(), lr=optimizer.param_groups[0]['lr'])

    avg_epoch_loss = epoch_loss / len(train_loader)
    epoch_train_acc = accuracy_score(train_targets_epoch, train_preds_epoch)

    training_results['loss_list'].append(avg_epoch_loss)
    training_results['acc_list'].append(epoch_train_acc)

    print(f"Epoch {epoch}/{cfg['epochs']} - Train Loss: {avg_epoch_loss:.4f}, Train Acc: {epoch_train_acc:.4f}, LR: {optimizer.param_groups[0]['lr']:.2e}")

    if epoch % cfg['val_epochs'] == 0:
        model.eval()
        val_loss = 0
        val_preds_epoch = []
        val_targets_epoch = []
        with torch.no_grad():
            progress_bar_val = tqdm(val_loader, desc=f"Epoch {epoch}/{cfg['epochs']} [Val]", leave=False)
            for data, target in progress_bar_val:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = criterion(output, target)
                val_loss += loss.item()
                preds = torch.argmax(output, dim=1)
                val_preds_epoch.extend(preds.cpu().numpy())
                val_targets_epoch.extend(target.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_acc = accuracy_score(val_targets_epoch, val_preds_epoch)
        val_f1 = f1_score(val_targets_epoch, val_preds_epoch, average='macro', zero_division=0)
        val_kappa = cohen_kappa_score(val_targets_epoch, val_preds_epoch)

        training_results['val_epoch_list'].append(epoch)
        training_results['val_acc_list'].append(val_acc)
        training_results['val_f1_macro_list'].append(val_f1)
        training_results['val_kappa_list'].append(val_kappa)

        print(f"  Validation - Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}, Val F1 (Macro): {val_f1:.4f}, Val Kappa: {val_kappa:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            if cfg.get('save_checkpoints', True):
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_acc': val_acc,
                    'config': cfg # 保存当时的配置 (注意：cfg可能包含不可序列化的对象，如scaler)
                }, best_model_path)
                print(f"   Best model saved to {best_model_path} with Val Acc: {best_val_acc:.4f}")
        
        if scheduler:
            if cfg['lr_scheduler_type'].lower() == 'plateau':
                scheduler.step(val_acc)
            elif cfg['lr_scheduler_type'].lower() != 'plateau' and scheduler is not None: # 避免对None调用step
                 scheduler.step()
    elif scheduler and cfg['lr_scheduler_type'].lower() != 'plateau' and scheduler is not None: # Step scheduler at each epoch if not plateau
        scheduler.step()

print("Training completed.")


In [ ]:
# ## 9. 可视化训练曲线
# (这部分与之前相同)

# %%
curves_filename = f"{cfg.get('model_name', 'model')}_{cfg.get('dataset_name','dataset')}_training_curves_mat_format.png"
curves_save_path = os.path.join(cfg.get('save_dir', './results'), curves_filename)
visualize_training_curves(training_results, save_path=curves_save_path)


In [ ]:
# ## 10. 在测试集上评估模型
# (这部分与之前相同)

# %%
if cfg.get('save_checkpoints', True) and os.path.exists(best_model_path):
    print(f"Loading best model from: {best_model_path}")
    checkpoint = torch.load(best_model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
else:
    print("No best model checkpoint found. Using the current model state for testing.")

model.eval()
test_preds_final = []
test_targets_final = []
with torch.no_grad():
    progress_bar_test = tqdm(test_loader, desc="Testing", leave=False)
    for data, target in progress_bar_test:
        data, target = data.to(device), target.to(device)
        output = model(data)
        preds = torch.argmax(output, dim=1)
        test_preds_final.extend(preds.cpu().numpy())
        test_targets_final.extend(target.cpu().numpy())

test_acc = accuracy_score(test_targets_final, test_preds_final)
test_f1_macro = f1_score(test_targets_final, test_preds_final, average='macro', zero_division=0)
test_f1_weighted = f1_score(test_targets_final, test_preds_final, average='weighted', zero_division=0)
test_kappa = cohen_kappa_score(test_targets_final, test_preds_final)

print("\nTest Set Results:")
print(f"  Accuracy: {test_acc:.4f}")
print(f"  F1 Score (Macro): {test_f1_macro:.4f}")
print(f"  F1 Score (Weighted): {test_f1_weighted:.4f}")
print(f"  Cohen's Kappa: {test_kappa:.4f}")


In [ ]:
# ## 11. 可视化混淆矩阵
# (这部分与之前相同)

# %%
conf_matrix = confusion_matrix(test_targets_final, test_preds_final)
cm_filename = f"{cfg.get('model_name', 'model')}_{cfg.get('dataset_name','dataset')}_confusion_matrix_mat_format.png"
cm_save_path = os.path.join(cfg.get('save_dir', './results'), cm_filename)
visualize_confusion_matrix(conf_matrix, save_path=cm_save_path, log_scale=True)

# %% [markdown]
# ## 12. 保存最终结果和配置 (可选)
# (这部分与之前相同, 注意处理不可序列化对象)

# %%
final_results = {
    'best_validation_accuracy': best_val_acc,
    'test_accuracy': test_acc,
    'test_f1_macro': test_f1_macro,
    'test_f1_weighted': test_f1_weighted,
    'test_kappa': test_kappa,
    'training_results': training_results,
    'hyperparameters_used': cfg.copy() # 保存实际使用的配置副本
}

# 清理不可序列化的配置项以便保存为JSON
if 'scaler' in final_results['hyperparameters_used']:
    del final_results['hyperparameters_used']['scaler']
if 'pca_model' in final_results['hyperparameters_used']: # 尽管MAT加载路径不使用PCA，以防万一
    del final_results['hyperparameters_used']['pca_model']
if 'device' in final_results['hyperparameters_used'] and not isinstance(final_results['hyperparameters_used']['device'], str):
     final_results['hyperparameters_used']['device'] = str(final_results['hyperparameters_used']['device'])


results_filename = f"{cfg.get('model_name', 'model')}_{cfg.get('dataset_name','dataset')}_final_results_mat_format.json"
results_save_path = os.path.join(cfg.get('save_dir', './results'), results_filename)

with open(results_save_path, 'w') as f:
    json.dump(final_results, f, indent=4)

print(f"Final results saved to: {results_save_path}")
print("Notebook execution completed.")